# 추론 및 프롬프트 엔지니어링

모델 로드 → 프롬프트 실험 → 결과 수집

In [ ]:
from google.colab import drive
import os

print("🔄 Google Drive 마운트 중...")
drive.mount('/content/drive')

# Drive에 저장 경로 생성
MODEL_CACHE = "/content/drive/MyDrive/colab_models"
LORA_CACHE = "/content/drive/MyDrive/colab_lora"

os.makedirs(MODEL_CACHE, exist_ok=True)
os.makedirs(LORA_CACHE, exist_ok=True)

print(f"✅ Google Drive 마운트 완료")
print(f"📁 모델 경로: {MODEL_CACHE}")
print(f"📁 LoRA 경로: {LORA_CACHE}")


In [ ]:
!pip install -q unsloth torch transformers peft huggingface-hub

print("✅ 설치 완료")

from google.colab import userdata
userdata.get('HF_TOKEN')

In [ ]:
from huggingface_hub import snapshot_download
import subprocess


MODEL_CACHE = "/content/models/colab_models"

print("🔄 모델 다운로드 중...\n")

# ========== Qwen3.5-9B ==========
# print("📥 Qwen3.5-9B 다운로드...")
# qwen_9b_path = snapshot_download(
#     repo_id="Qwen/Qwen3.5-9B",
#     cache_dir=MODEL_CACHE,
#     force_download=False,
# )
qwen_9b_path = snapshot_download(
    repo_id="Qwen/Qwen3.5-9B",
    cache_dir=MODEL_CACHE,
    local_files_only=True,
)

print(f"✅ 저장됨: {qwen_9b_path}\n")

# ========== Qwen2.5-7B-Instruct ==========
print("📥 Qwen2.5-7B-Instruct 다운로드...")
# qwen_7b_path = snapshot_download(
#     repo_id="Qwen/Qwen2.5-7B-Instruct",
#     cache_dir=MODEL_CACHE,
#     force_download=False,
qwen_7b_path = snapshot_download(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    cache_dir=MODEL_CACHE,
    local_files_only=True,
)
print(f"✅ 저장됨: {qwen_7b_path}\n")

# 용량 확인
result = subprocess.run(["du", "-sh", MODEL_CACHE], capture_output=True, text=True)
print(f"📦 총 용량: {result.stdout.strip()}")

# 경로 저장 (이후 Cell에서 사용)
model_paths = {
    "qwen_9b": qwen_9b_path,
    "qwen_7b": qwen_7b_path,
}

print("\n✅ 모델 다운로드 완료!")

In [ ]:
import os

lora_root = "/content/drive/MyDrive/colab_lora/models--teems79--interview-lora/snapshots/111f308453491011fd6ef25b26ac717f5cdbe2fd"

print("🔄 LoRA 경로 설정 중...\n")

# ========== Qwen3.5-9B LoRA ==========
lora_9b_pressure = os.path.join(lora_root, "qwen3.5", "pressure")
lora_9b_friendly = os.path.join(lora_root, "qwen3.5", "friendly")

print(f"✅ Qwen3.5-9B Pressure: {lora_9b_pressure}")
print(f"✅ Qwen3.5-9B Friendly: {lora_9b_friendly}\n")

# ========== Qwen2.5-7B LoRA (보너스) ==========
lora_7b_pressure = os.path.join(lora_root, "qwen2.5", "pressure")
lora_7b_friendly = os.path.join(lora_root, "qwen2.5", "friendly")

print(f"✅ Qwen2.5-7B Pressure: {lora_7b_pressure}")
print(f"✅ Qwen2.5-7B Friendly: {lora_7b_friendly}\n")

# ========== 경로 저장 ==========
lora_paths = {
    "qwen_9b": {
        "pressure": lora_9b_pressure,
        "friendly": lora_9b_friendly,
    },
    "qwen_7b": {
        "pressure": lora_7b_pressure,
        "friendly": lora_7b_friendly,
    },
}

# 확인
import subprocess
for model, personas in lora_paths.items():
    for persona, path in personas.items():
        if os.path.exists(path):
            result = subprocess.run(["du", "-sh", path], capture_output=True, text=True)
            print(f"✅ {model} {persona}: {result.stdout.strip()}")
        else:
            print(f"❌ {model} {persona} 없음")

print("\n✅ 모든 LoRA 경로 설정 완료!")

In [ ]:
import torch
from unsloth import FastLanguageModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import PeftModel

# ✅ 4bit 양자화 설정 객체로 분리
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

_models_cache = {}

def load_model(model_name: str, persona: str = None):
    """모델 로드 (Google Drive에서)"""

    cache_key = f"{model_name}_{persona}" if persona else model_name
    print(f"🔍 cache_key: {cache_key}, 캐시 현황: {list(_models_cache.keys())}")  # ✅ 추가
    if cache_key in _models_cache:
        print(f"✅ [{model_name}] 캐시에서 로드")
        return _models_cache[cache_key]

    print(f"\n🔄 [{model_name}] 로드 중...")

    # ── Qwen3.5-9B ──
    if model_name == "qwen_9b":
        try:
            model, _ = FastLanguageModel.from_pretrained(
                model_name=model_paths["qwen_9b"],
                max_seq_length=2048,
                dtype=torch.bfloat16,
                load_in_4bit=True,
            )
            # ✅ Processor 대신 AutoTokenizer 강제 사용
            tokenizer = AutoTokenizer.from_pretrained(
                model_paths["qwen_9b"],
                trust_remote_code=True,
            )
            if persona:
                model = PeftModel.from_pretrained(model, lora_paths["qwen_9b"][persona])
                model = model.merge_and_unload()
            FastLanguageModel.for_inference(model)

        except Exception as e:
            print(f"⚠️ Unsloth 실패, 기본 로드로 전환: {e}")
            # ✅ quantization_config으로 전달 (load_in_4bit 직접 전달 제거)
            model = AutoModelForCausalLM.from_pretrained(
                model_paths["qwen_9b"],
                quantization_config=bnb_config,
                device_map="auto",
                torch_dtype=torch.bfloat16,
            )
            tokenizer = AutoTokenizer.from_pretrained(
                model_paths["qwen_9b"],
                trust_remote_code=True,
            )
            if persona:
                model = PeftModel.from_pretrained(model, lora_paths["qwen_9b"][persona])
                model = model.merge_and_unload()

    # ── Qwen2.5-7B ──
    elif model_name == "qwen_7b":
        try:
            model = AutoModelForCausalLM.from_pretrained(
                model_paths["qwen_7b"],
                quantization_config=bnb_config,  # ✅ 동일하게 적용
                device_map="auto",
                torch_dtype=torch.bfloat16,
            )
            tokenizer = AutoTokenizer.from_pretrained(
                model_paths["qwen_7b"],
                trust_remote_code=True,
            )
            if persona:
                model = PeftModel.from_pretrained(model, lora_paths["qwen_7b"][persona])
                model = model.merge_and_unload()

        except Exception as e:
            print(f"⚠️ LoRA 적용 실패: {e}")
            model = AutoModelForCausalLM.from_pretrained(
                model_paths["qwen_7b"],
                quantization_config=bnb_config,
                device_map="auto",
                torch_dtype=torch.bfloat16,
            )
            tokenizer = AutoTokenizer.from_pretrained(
                model_paths["qwen_7b"],
                trust_remote_code=True,
            )

    model.eval()
    _models_cache[cache_key] = (model, tokenizer)
    print(f"✅ [{model_name}] 로드 완료")
    return model, tokenizer

In [ ]:
import torch
import time
import re


SYSTEM_PROMPTS = {
    "qwen_9b": {
        "pressure": "너는 ICT 직무 압박 면접관이다.",
        "friendly": "너는 ICT 직무 코칭형 면접관이다."
    },

    "qwen_7b": {
        "pressure": "너는 ICT 직무 압박 면접관이다.",
        "friendly": "너는 ICT 직무 코칭형 면접관이다."
    }
}

STATE_RULES = {
    "FIRST_QUESTION": [
        "지원자는 신입이다.",
        "직무 관심과 간단한 경험 확인 수준으로만 질문한다.",
        "기술 깊이, 성과, 수치 질문 금지.",
        "가볍게 무엇을 해봤는지 정도만 묻는다.",
        "예시 수준: 어떤 개발 경험이 있으신지 간단히 말씀해 주세요.",
        "예시 수준: ICT 직무에 관심을 갖게 된 계기가 있으신가요?"
    ],

    "FOLLOWUP": [
        "이전 답변의 키워드 1개만 사용한다.",
        "새로운 주제 추가 금지.",
        "깊이 확장 질문만 허용한다."
    ]
}



PRESSURE_STYLE = "\n".join(f"- {r}" for r in [
    "첫 문장에는 지원자 답변의 핵심 키워드 1개를 포함하고, '모호', '정확히', '구체적이지', '명확하지' 중 1개를 함께 사용하여 답변의 부족한 지점을 지적한다.",
    "긍정 표현 및 이모지를 사용하지 않는다.",
    "발화 길이는 3문장 이하로 끝낸다.",
    "마지막 질문 문장에는 '왜', '어떤', '어떻게', '구체적으로', '근거', '수치', '사례', '경험' 중 1개를 포함한다.",
    "인정·완충 표현을 사용하지 않는다.",
])

FRIENDLY_STYLE = "\n".join(f"- {r}" for r in [
    "인정 또는 완충 표현을 발화 전체에 1회 이상 포함한다.",
    "'특히', '예를 들어', '구체적인 예로', '어떠한 배경에서' 중 1개 이상을 반드시 포함한다.",
    "유도 키워드를 사용하되 질문의 개수는 반드시 1개로 유지한다.",
    "'상황', '역할', '행동', '결과', '이유', '근거', '사례', '배운 점' 중 1개 이상을 포함한다.",
    "발화 길이는 4문장 이하로 끝낸다.",
    "직접적인 부정 평가 표현을 사용하지 않는다.",
])


# ==============================================================================
# ==============================================================================

COMMON_RULES = [
    "반드시 존댓말을 사용한다.",
    "마지막 문장은 질문형으로 끝낸다.",
    "무례한 인신공격, 비하, 차별 표현을 사용하지 않는다.",
    "예시 문장을 그대로 복사하지 않고 입력 답변에 맞게 표현을 바꾼다.",
    "질문은 1개만 생성한다.",
    "한국어만 사용한다.",
]

# COMMON_MIN = [
#     "한국어로만 출력",
#     "질문 1개만 생성"
# ]

PRESSURE_V1 = """
당신은 ICT 직무 압박 면접관입니다.

목표:
지원자의 역량과 경험을 검증하기 위한 면접 질문을 생성합니다.


공통 규칙:
{chr(10).join(f"- {r}" for r in COMMON_RULES)}
"""

# 공통 규칙:
# {chr(10).join(f"- {r}" for r in COMMON_MIN)}

FRIENDLY_V1 = """
당신은 ICT 직무 코칭형 면접관입니다.

목표:
지원자의 역량과 경험을 검증하기 위한 면접 질문을 생성합니다.

공통 규칙:
{chr(10).join(f"- {r}" for r in COMMON_RULES)}
"""

# 공통 규칙:
# {chr(10).join(f"- {r}" for r in COMMON_MIN)}

import re

# ── 상태 감지 ──
def get_state_from_prompt(prompt: str) -> str:
    if "FIRST_QUESTION" in prompt:
        return "FIRST_QUESTION"
    return "FOLLOWUP"

def contains_chinese(text: str) -> bool:
    return bool(re.search(r"[\u4e00-\u9fff]", text))

# ── Validator ──

def validate_pressure_checks(text: str) -> dict:
    stripped = text.strip()
    sentences = [s for s in re.split(r"[.!?]", text) if s.strip()]

    return {
        "p1_keyword": any(
            k in text
            for k in PRESSURE_RULES["must_include_keywords_any"][0]
        ),
        "p2_no_forbidden": not any(
            p in text
            for p in PRESSURE_RULES["forbidden_phrases"]
        ),
        "p3_ends_question": stripped.endswith("?"),
        "p4_no_emoji": not re.search(
            r"[\U00010000-\U0010ffff]|[\u2600-\u27BF]",
            text
        ),
        "p5_sentence_limit": len(sentences) <= PRESSURE_RULES["max_sentences"],
    }

def validate_pressure(text: str) -> bool:
    score = 0

    # 1. 한국어 비율
    if korean_ratio(text) >= PRESSURE_RULES["min_korean_ratio"]:
        score += 1

    # 2. 질문 1개
    if text.count("?") <= PRESSURE_RULES["max_questions"]:
        score += 1

    # 3. 질문 키워드 포함
    if any(k in text for k in PRESSURE_RULES["must_include_keywords_any"][0]):
        score += 1

    # 4. forbidden_phrases 없음
    if not any(p in text for p in PRESSURE_RULES["forbidden_phrases"]):
        score += 1

    # 5. 마지막 문장이 ?로 끝남
    stripped = text.strip()
    if stripped.endswith("?"):
        score += 1

    # 6. 이모지 없음
    if not re.search(r"[\U00010000-\U0010ffff]|[\u2600-\u27BF]", text):
        score += 1

    # 7. 문장 수 이하
    sentences = [s for s in re.split(r"[.!?]", text) if s.strip()]
    if len(sentences) <= PRESSURE_RULES["max_sentences"]:
        score += 1

    return score >= 5  # 7개 중 5개 이상


def validate_friendly_checks(text: str) -> dict:
    stripped = text.strip()
    sentences = [s for s in re.split(r"[.!?]", text) if s.strip()]

    return {
        "f1_context_expansion": any(
            k in text
            for k in FRIENDLY_RULES["must_include_one_of"]["context_expansion"]
        ),
        "f2_no_forbidden": not any(
            p in text
            for p in FRIENDLY_RULES["forbidden_phrases"]
        ),
        "f3_softener": any(
            s in text
            for s in FRIENDLY_RULES["must_include_softener"]
        ),
        "f4_ends_question": stripped.endswith("?"),
        "f5_sentence_limit": len(sentences) <= FRIENDLY_RULES["max_sentences"],
    }

def validate_friendly(text: str) -> bool:
    score = 0

    # 1. 한국어 비율
    if korean_ratio(text) >= FRIENDLY_RULES["min_korean_ratio"]:
        score += 1

    # 2. 질문 정확히 1개
    if text.count("?") == FRIENDLY_RULES["exact_questions"]:
        score += 1

    # 3. context_expansion 키워드 포함
    if any(k in text for k in FRIENDLY_RULES["must_include_one_of"]["context_expansion"]):
        score += 1

    # 4. forbidden_phrases 없음
    if not any(p in text for p in FRIENDLY_RULES["forbidden_phrases"]):
        score += 1

    # 5. 완충 표현 포함
    if any(s in text for s in FRIENDLY_RULES["must_include_softener"]):
        score += 1

    # 6. 마지막 문장이 ?로 끝남
    stripped = text.strip()
    if stripped.endswith("?"):
        score += 1

    # 7. 문장 수 이하
    sentences = [s for s in re.split(r"[.!?]", text) if s.strip()]
    if len(sentences) <= FRIENDLY_RULES["max_sentences"]:
        score += 1

    return score >= 5  # 7개 중 5개 이상

def hard_fail(text, threshold=0.8):
    if count_questions(text) != 1:
        return True, "question_count"

    if korean_ratio(text) < threshold:
        return True, "low_korean_ratio"

    if contains_chinese(text):
        return True, "contains_chinese"

    return False, ""

def validate_output(text: str, persona: str, current_state: str = None, threshold: float = 0.8) -> dict:
    failed, reason = hard_fail(text, threshold)

    if failed:
        return {
            "valid": False,
            "hard_fail": True,
            "hard_fail_reason": reason,
            "style_score": 0,
            "checks": {},
        }

    if persona == "pressure":
        checks = validate_pressure_checks(text)
    elif persona == "friendly":
        checks = validate_friendly_checks(text)
    else:
        return {
            "valid": False,
            "hard_fail": True,
            "hard_fail_reason": "unknown_persona",
            "style_score": 0,
            "checks": {},
        }

    passed = sum(checks.values())
    total = len(checks)
    style_score = round(passed / total * 100, 2) if total else 0

    return {
        "valid": style_score >= 80,
        "hard_fail": False,
        "hard_fail_reason": "",
        "style_score": style_score,
        "checks": checks,
    }

# ── 추론 함수 ──
import time

def generate_once(model, tokenizer, messages, max_tokens=100, temperature=0.7):

    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except Exception:
        # enable_thinking 미지원 토크나이저 fallback
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    response = re.sub(r"^Q:\s*", "", response)
    response = re.sub(r"^질문:\s*", "", response)
    return response

def generate_with_retry(model, tokenizer, messages, persona, current_state, max_retry=2):
    last_output = ""

    for attempt in range(max_retry):
        output = generate_once(
            model,
            tokenizer,
            messages,
            temperature=0.7 + attempt * 0.1,
        )

        last_output = output
        eval_result = validate_output(output, persona, current_state)

        if eval_result["valid"]:
            return output

        print(
            f"  [RETRY {attempt+1}] 규칙 위반 → 재생성 "
            f"(persona={persona}, "
            f"hard_fail={eval_result['hard_fail']}, "
            f"reason={eval_result['hard_fail_reason']}, "
            f"style_score={eval_result['style_score']})"
        )

    return last_output

def generate(model, tokenizer, prompt, model_name, persona, max_tokens=100):
    start         = time.time()
    current_state = get_state_from_prompt(prompt)
    system_msg    = build_system_msg(persona, current_state)

    # ✅ 안전장치
    if not prompt or not prompt.strip():
        print("⚠️ 빈 prompt 감지 → 스킵")
        return "", 0.0

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": prompt},
    ]

    response = generate_with_retry(model, tokenizer, messages, persona, current_state, max_retry=2)
    elapsed  = time.time() - start
    return response, elapsed

print("✅ Validator + 추론 함수 설정 완료")

In [ ]:
import gc
import torch

_models_cache.clear()
torch.cuda.empty_cache()
gc.collect()

print(f"GPU: {torch.cuda.memory_allocated()/1024**3:.1f} / {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

In [ ]:
def korean_ratio(text: str) -> float:
    if not text:
        return 0.0

    korean_chars = len(re.findall(r"[가-힣]", text))
    total_chars = len(re.findall(r"\S", text))  # 공백 제외 문자 수

    if total_chars == 0:
        return 0.0

    return korean_chars / total_chars

In [ ]:
# ============================================================
# Cell 9 : 테스트 프롬프트
# ============================================================
test_prompts = [
"""
현재 면접 상태: FIRST_QUESTION
지원자는 신입 개발자입니다.
기본 경험 확인 수준의 질문을 생성하세요.
직무: ICT
지원자 답변은 아직 존재하지 않습니다.
첫 번째 면접 질문을 생성하세요.
규칙:
- 답변 지적 금지
- 후속 질문 금지
- 질문 1개만 생성
""",
"""
현재 면접 상태: FOLLOWUP
직무: ICT
이전 질문: 자기소개 부탁드립니다.
지원자 답변: 저는 Python을 배웠습니다.
위 답변을 기반으로 후속 질문 1개를 생성하세요.
""",
"""
현재 면접 상태: FOLLOWUP
직무: ICT
이전 질문: 진행했던 프로젝트를 설명해주세요.
지원자 답변: 쇼핑몰 웹사이트를 개발했습니다.
위 답변을 기반으로 후속 질문 1개를 생성하세요.
""",
"""
현재 면접 상태: FOLLOWUP
직무: ICT
이전 질문: 팀 프로젝트에서 어떤 역할을 담당했나요?
지원자 답변: 백엔드를 담당했습니다.
위 답변을 기반으로 후속 질문 1개를 생성하세요.
""",
"""
현재 면접 상태: FOLLOWUP
직무: ICT
이전 질문: API를 사용해본 경험이 있나요?
지원자 답변: 공공데이터 API를 사용했습니다.
위 답변을 기반으로 후속 질문 1개를 생성하세요.
""",
"""
현재 면접 상태: FOLLOWUP
직무: ICT
이전 질문: 데이터베이스 사용 경험을 설명해주세요.
지원자 답변: MySQL을 사용했습니다.
위 답변을 기반으로 후속 질문 1개를 생성하세요.
""",
"""
현재 면접 상태: FOLLOWUP
직무: ICT
이전 질문: 개발 중 어려웠던 경험이 있나요?
지원자 답변: API 연동이 어려웠습니다.
위 답변을 기반으로 후속 질문 1개를 생성하세요.
""",
"""
현재 면접 상태: FOLLOWUP
직무: ICT
이전 질문: 협업 경험을 설명해주세요.
지원자 답변: 팀원들과 GitHub를 사용했습니다.
위 답변을 기반으로 후속 질문 1개를 생성하세요.
""",
"""
현재 면접 상태: FOLLOWUP
직무: ICT
이전 질문: 성능 개선 경험이 있나요?
지원자 답변: SQL을 최적화했습니다.
위 답변을 기반으로 후속 질문 1개를 생성하세요.
""",
"""
현재 면접 상태: FOLLOWUP
직무: ICT
이전 질문: AI 프로젝트 경험이 있나요?
지원자 답변: 챗봇을 만들었습니다.
위 답변을 기반으로 후속 질문 1개를 생성하세요.
""",
"""
현재 면접 상태: FOLLOWUP
직무: ICT
이전 질문: 클라우드 사용 경험이 있나요?
지원자 답변: AWS를 사용했습니다.
위 답변을 기반으로 후속 질문 1개를 생성하세요.
""",
# 모호한 답변
"""
현재 면접 상태: FOLLOWUP
직무: ICT
이전 질문: 본인의 강점은 무엇인가요?
지원자 답변: 문제 해결을 잘 합니다.
위 답변을 기반으로 후속 질문 1개를 생성하세요.
""",
# 답변 회피
"""
현재 면접 상태: FOLLOWUP
직무: ICT
이전 질문: 해당 프로젝트에서 실패한 경험이 있나요?
지원자 답변: 특별히 실패한 경험은 없습니다.
위 답변을 기반으로 후속 질문 1개를 생성하세요.
""",
# 긴 답변
"""
현재 면접 상태: FOLLOWUP
직무: ICT
이전 질문: 진행했던 프로젝트를 설명해주세요.
지원자 답변: 저는 팀 프로젝트로 음식 배달 앱을 개발했습니다. 프론트엔드는 React, 백엔드는 Spring Boot를 사용했고 AWS EC2에 배포했습니다. 저는 주로 백엔드 API 설계를 맡았고 팀원 3명과 협업했습니다. 개발 기간은 3개월이었습니다.
위 답변을 기반으로 후속 질문 1개를 생성하세요.
""",
]

print(f"✅ 테스트 프롬프트 {len(test_prompts)}개 설정 완료")

In [ ]:
# ============================================================
# Cell 10 : 실험 실행
# ============================================================
import pandas as pd
from datetime import datetime

results = []

def run_and_collect(model_name, persona):
    model, tokenizer = load_model(model_name, persona)

    for idx, prompt in enumerate(test_prompts, 1):
        current_state = get_state_from_prompt(prompt)

        resp, elapsed = generate(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            model_name=model_name,
            persona=persona,
        )

        eval_result = validate_output(resp, persona, current_state)

        results.append({
            "model": model_name,
            "persona": persona,
            "test_id": idx,
            "prompt": prompt,
            "state": current_state,
            "response": resp,
            "time": round(elapsed, 2),
            "korean_ratio": round(korean_ratio(resp), 3),
            "valid": eval_result["valid"],
            "hard_fail": eval_result["hard_fail"],
            "hard_fail_reason": eval_result["hard_fail_reason"],
            "style_score": eval_result["style_score"],
            "checks": eval_result["checks"],
        })

    # 메모리 해제
    del model, tokenizer
    cache_key = f"{model_name}_{persona}"
    if cache_key in _models_cache:
        del _models_cache[cache_key]
    torch.cuda.empty_cache()
    gc.collect()
    print(f"✅ [{model_name} / {persona}] 완료 및 메모리 해제")


print("\n" + "="*80)
print("🎤 4가지 모델 비교 분석 (순차 로드)")
print("="*80)

run_and_collect("qwen_9b", "pressure")
run_and_collect("qwen_9b", "friendly")
run_and_collect("qwen_7b", "pressure")
run_and_collect("qwen_7b", "friendly")

print("\n✅ 전체 실행 완료")

# ============================================================
# Cell 11 : 결과 출력 + CSV 저장
# ============================================================
df = pd.DataFrame(results)

for idx, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*80}")
    print(f"📌 질문 {idx}: {prompt.strip()[:60]}...")
    print("="*80)
    for model_name, persona, label in [
        ("qwen_9b", "pressure", "Qwen3.5-9B - Pressure"),
        ("qwen_9b", "friendly", "Qwen3.5-9B - Friendly"),
        ("qwen_7b", "pressure", "Qwen2.5-7B - Pressure"),
        ("qwen_7b", "friendly", "Qwen2.5-7B - Friendly"),
    ]:
        row = df[
            (df["model"]   == model_name) &
            (df["persona"] == persona) &
            (df["test_id"] == idx)
        ].iloc[0]
        valid_mark = "✅" if row["valid"] else "❌"
        print(f"\n🎤 [{label}] {valid_mark} (kr={row['korean_ratio']:.2f})")
        print(f"응답: {row['response']}")
        print(f"⏱️  {row['time']:.2f}초")

print("\n✅ 비교 분석 완료!")

# CSV 저장
timestamp  = datetime.now().strftime("%Y%m%d_%H%M")
local_path = f"baseline_v12_{timestamp}.csv"
drive_path = f"/content/drive/MyDrive/prompt_experiment/baseline_v12_{timestamp}.csv"

os.makedirs("/content/drive/MyDrive/prompt_experiment", exist_ok=True)
df.to_csv(local_path, index=False, encoding="utf-8-sig")
df.to_csv(drive_path, index=False, encoding="utf-8-sig")

print(f"\nCSV 저장 완료!")
print(f"  로컬 : {local_path}")
print(f"  Drive: {drive_path}")

In [ ]:
def get_history_text(history):
    lines = []
    for item in history[-3:]:
        lines.append(f"Q: {item['question']}")
        if item['answer']:
            lines.append(f"A: {item['answer']}")
    return "\n".join(lines)

def run_interview(model_name: str, persona: str = None, job_role: str = "ICT"):
    """대화형 면접 시뮬레이터"""

    model, tokenizer = load_model(model_name, persona)
    history = []
    turn = 0

    model_display = f"{model_name.upper()} {persona}" if persona else model_name.upper()

    print("\n" + "="*70)
    print(f"🎤 면접 시뮬레이터 | 모델: {model_display} | 직무: {job_role}")
    print("="*70)
    print("💡 명령: '시작' = 시작 | '종료' = 종료 | '모델변경' = 다른 모델\n")

    while True:
        user_input = input("👤 지원자: ").strip()

        if not user_input:
            continue

        # 종료
        if any(t in user_input for t in ["종료", "끝", "나가"]):
            print(f"\n✅ 면접 종료 ({turn}턴)\n")
            break

        # 모델 변경
        if "모델변경" in user_input:
            print("\n📌 사용 가능한 모델:")
            print("  1. Qwen3.5-9B (Pressure)")
            print("  2. Qwen3.5-9B (Friendly)")
            print("  3. Qwen2.5-7B (Pressure)")
            print("  4. Qwen2.5-7B (Friendly)")
            choice = input("선택 (1-4): ").strip()
            mapping = {
                "1": ("qwen_9b", "pressure"),
                "2": ("qwen_9b", "friendly"),
                "3": ("qwen_7b", "pressure"),
                "4": ("qwen_7b", "friendly"),
            }
            if choice in mapping:
                return run_interview(*mapping[choice], job_role)
            else:
                print("❌ 잘못된 선택\n")
            continue

        # 시작
        if any(t in user_input for t in ["시작"]):
            turn = 0
            history = []
            prompt = f"{job_role} 직무 면접을 시작합니다. 지원자에게 첫 번째 질문을 생성하세요. 이전 답변이 없으므로 답변 지적 없이 질문만 생성하세요."

            response, elapsed = generate(model, tokenizer, prompt, model_name, persona)

            print(f"\n🎤 [면접관]: {response}")
            print(f"   ⏱️ {elapsed:.2f}초\n")

            history.append({"question": response, "answer": None})
            continue

        # 진행
        if history and history[-1]["answer"] is None:
            history[-1]["answer"] = user_input
            turn += 1

            if turn >= 10:
                print("\n🎤 [면접관]: 수고하셨습니다.\n")
                break

            recent = get_history_text(history)
            prompt = f"""이전 대화:
{recent}

위 지원자의 답변을 바탕으로 다음 면접 질문을 생성하세요."""

            response, elapsed = generate(model, tokenizer, prompt, model_name, persona)

            print(f"\n🎤 [면접관]: {response}")
            print(f"   ⏱️ {elapsed:.2f}초\n")

            history.append({"question": response, "answer": None})

        else:
            print("  ℹ️ '시작'을 입력해주세요.\n")

# 모델 선택
print("\n" + "="*70)
print("📌 사용할 모델 선택")
print("="*70)
print("1. Qwen3.5-9B (Pressure)")
print("2. Qwen3.5-9B (Friendly)")
print("3. Qwen2.5-7B (Pressure) ⭐ 새로 추가!")
print("4. Qwen2.5-7B (Friendly) ⭐ 새로 추가!\n")

choice = input("선택 (1-4): ").strip()

if choice == "1":
    run_interview("qwen_9b", "pressure", "ICT")
elif choice == "2":
    run_interview("qwen_9b", "friendly", "ICT")
elif choice == "3":
    run_interview("qwen_7b", "pressure", "ICT")
elif choice == "4":
    run_interview("qwen_7b", "friendly", "ICT")
else:
    print("❌ 잘못된 선택")